# Project - Airline AI Assistant

We'll now bring together what we've learned to make an AI Customer Support assistant for an Airline

In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
# Initialization

load_dotenv(override=True)


MODEL = "gemma4:e4b"
ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')


In [3]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [4]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Tools



In [ ]:


ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"Tool called for city {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "Unknown ticket price")
    return f"The price of a ticket to {destination_city} is {price}"


In [6]:
get_ticket_price("London")

Tool called for city London


'The price of a ticket to London is $799'

In [7]:
get_ticket_price("Liverpool")

Tool called for city Liverpool


'The price of a ticket to Liverpool is Unknown ticket price'

In [8]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [9]:
# And this is included in a list of tools:

tools = [{"type": "function", "function": price_function}]

In [10]:
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

## Getting OpenAI to use our Tool



In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    print(response.choices[0])

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        response = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        response = ollama.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [ ]:


def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    if tool_call.function.name == "get_ticket_price":
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get('destination_city')
        price_details = get_ticket_price(city)
        response = {
            "role": "tool",
            "content": price_details,
            "tool_call_id": tool_call.id
        }
    return response

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


ChatCompletion(id='chatcmpl-33', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hello! How can I help you with your flight booking today?', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning='1.  **Analyze the user input:** The user said, "hi there." This is a casual greeting.\n2.  **Determine the goal/context:** I am an assistant for "FlightAI," delivering short, courteous, and accurate answers (max 1 sentence).\n3.  **Identify necessary tools:** No tools are needed as the request is only a greeting.\n4.  **Formulate the response:** A simple, friendly greeting that aligns with the persona and constraints.\n\n*Self-Correction/Refinement:* Ensure the tone is courteous and professional yet friendly. Keep it conversational but direct.\n\n5.  **Final Answer Generation:** Welcome them in a 1-sentence format.'))], created=1782839959, model='gemma4:e4b', object='chat.completion', se

## Some improvements

Handling multiple tool calls in 1 response

Handling multiple tool calls 1 after another

In [16]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = ollama.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [20]:
def handle_tool_calls(message):
    responses = []
    print(message.tool_calls)
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [23]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


[ChatCompletionMessageFunctionToolCall(id='call_o54h6r6c', function=Function(arguments='{"destination_city":"Paris"}', name='get_ticket_price'), type='function', index=0)]
Tool called for city Paris
[ChatCompletionMessageFunctionToolCall(id='call_v363pzx4', function=Function(arguments='{"destination_city":"Berlin"}', name='get_ticket_price'), type='function', index=0)]
Tool called for city Berlin
[ChatCompletionMessageFunctionToolCall(id='call_ljk1jcfb', function=Function(arguments='{"destination_city":"Tokyo"}', name='get_ticket_price'), type='function', index=0)]
Tool called for city Tokyo


In [22]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = ollama.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [24]:
import sqlite3


In [25]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

In [26]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [27]:
get_ticket_price("London")

DATABASE TOOL CALLED: Getting price for London


'No price data available for this city'

In [28]:
def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

In [29]:
ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999,"jakarta":699,"moscow":1399}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

In [30]:
get_ticket_price("Tokyo")

DATABASE TOOL CALLED: Getting price for Tokyo


'Ticket price to Tokyo is $1420.0'

In [31]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


[ChatCompletionMessageFunctionToolCall(id='call_hms8xegq', function=Function(arguments='{"destination_city":"Jakarta"}', name='get_ticket_price'), type='function', index=0)]
DATABASE TOOL CALLED: Getting price for Jakarta
[ChatCompletionMessageFunctionToolCall(id='call_0wld7i7v', function=Function(arguments='{"destination_city":"Paris"}', name='get_ticket_price'), type='function', index=0)]
DATABASE TOOL CALLED: Getting price for Paris
[ChatCompletionMessageFunctionToolCall(id='call_e5he8rhh', function=Function(arguments='{"destination_city":"Sydney"}', name='get_ticket_price'), type='function', index=0)]
DATABASE TOOL CALLED: Getting price for Sydney
